In [1]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv("../data/nfl_historico_clean.csv")

K_FACTOR = 20
HFA = 65
ELO_INICIAL = 1500

equipos = pd.concat([df['home_team'], df['away_team']]).unique()
ratings = {equipo: ELO_INICIAL for equipo in equipos}

def calcular_probabilidad(rating_local, rating_visita):
    diferencia = (rating_local + HFA) - rating_visita
    return 1 / (1 + 10 ** (-diferencia / 400))

def regresion_media(rating):
    return (2/3 * rating) + (1/3 * ELO_INICIAL)

In [5]:
historial_elo = []
temporada_actual = df['season'].min()

for index, row in df.iterrows():
    if row["season"] > temporada_actual:
        for equipo in ratings:
            ratings[equipo] = regresion_media(ratings[equipo])
        temporada_actual= row["season"]

    equipo_local = row['home_team']
    equipo_visita = row['away_team']

    elo_local_pre = ratings[equipo_local]
    elo_visita_pre = ratings[equipo_visita]

    prob_local = calcular_probabilidad(elo_local_pre, elo_visita_pre)
    prob_visita = 1 - prob_local

    if row['result'] > 0:
        res_local, res_visita = 1, 0
    elif row['result'] < 0:
        res_local, res_visita = 0, 1
    else:
        res_local, res_visita = 0.5, 0.5

    ratings[equipo_local] = elo_local_pre + K_FACTOR * (res_local - prob_local)
    ratings[equipo_visita] = elo_visita_pre + K_FACTOR * (res_visita - prob_visita)

    registro = row.to_dict()
    registro['elo_local_pre'] = round(elo_local_pre, 2)
    registro['elo_visita_pre'] = round(elo_visita_pre, 2)
    registro['prob_local'] = round(prob_local, 4)
    registro['elo_local_post'] = round(ratings[equipo_local], 2)
    registro['elo_visita_post'] = round(ratings[equipo_visita], 2)
    
    historial_elo.append(registro)

df_historial = pd.DataFrame(historial_elo)
print("✅ Procesamiento terminado. Los ratings finales de 2025 están listos.")
    

✅ Procesamiento terminado. Los ratings finales de 2025 están listos.


In [7]:
df_historial.head(-10)

,season,week,game_type,home_team,away_team,home_score,away_score,result,elo_local_pre,elo_visita_pre,prob_local,elo_local_post,elo_visita_post
0,2005,1,REG,NE,LV,30.0,20.0,10.0,1611.03,1415.66,0.8174,1614.68,1412.01
1,2005,1,REG,BUF,HOU,22.0,7.0,15.0,1496.63,1467.93,0.6317,1504.00,1460.56
2,2005,1,REG,CAR,NO,20.0,23.0,-3.0,1527.82,1485.42,0.6498,1514.82,1498.42
3,2005,1,REG,CLE,CIN,13.0,27.0,-14.0,1452.55,1552.67,0.4496,1443.56,1561.67
4,2005,1,REG,JAX,SEA,26.0,14.0,12.0,1400.86,1575.96,0.3466,1413.92,1562.89
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5683,2025,18,REG,PHI,WAS,17.0,24.0,-7.0,1617.30,1449.95,0.7921,1601.46,1465.79
5684,2025,18,REG,PIT,BAL,26.0,24.0,2.0,1530.50,1544.59,0.5728,1539.05,1536.04
5685,2025,19,WC,CAR,LA,31.0,34.0,-3.0,1441.83,1572.41,0.4067,1433.69,1580.54
5686,2025,19,WC,CHI,GB,31.0,27.0,4.0,1511.06,1536.76,0.5563,1519.94,1527.89


In [ ]:
df_validar = df_historial[df_historial['result'] != 0].copy()

# Predicción del modelo: True si el local era favorito y ganó, o no era favorito y perdió
df_validar['prediccion_correcta'] = (
    ((df_validar['prob_local'] > 0.5) & (df_validar['result'] > 0)) |
    ((df_validar['prob_local'] < 0.5) & (df_validar['result'] < 0))
)

# Calcular acierto global
acierto_global = df_validar['prediccion_correcta'].mean() * 100
print(f"🎯 Precisión Global Histórica (2005-2025): {acierto_global:.2f}%")

# Calcular acierto en una temporada específica (ej. 2023)[cite: 1].
acierto_2025 = df_validar[df_validar['season'] == 2025]['prediccion_correcta'].mean() * 100
print(f"🎯 Precisión Temporada 2025: {acierto_2025:.2f}%")

# Guardar la tabla final con los ratings calculados
df_historial.to_csv("../data/nfl_historico_con_elo.csv", index=False)
print("💾 Historial con cálculo Elo guardado en 'data/nfl_historico_con_elo.csv'")

🎯 Precisión Global Histórica (2005-2025): 61.79%
🎯 Precisión Temporada 2025: 60.21%
💾 Historial con cálculo Elo guardado en 'data/nfl_historico_con_elo.csv'
